## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

g:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "gpt-5-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2594.32it/s]


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='4ecda3d3-9749-4a47-a032-1902ce95e6e3', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery could refer to many things. Do you mean a person, a fictional character, or a brand?\n\nHere’s a quick overview:\n\n- Given name (unisex): Avery is a common first name for both men and women, often derived from Aubrey/Albery and sometimes explained as “elf ruler” or “noble ruler.”\n- Surname: Avery is also used as a last name.\n\nNotable Averys (examples):\n- Avery Brooks — American actor and director, best known for playing Captain Benjamin Sisko in Star Trek: Deep Space Nine.\n- Avery Bradley — American professional basketball player.\n- Avery Singer — Contemporary American artist.\n- Avery Dulles — American Jesuit priest and theologian, author of influential works like Models of the Church.\n- Sgt. Avery Johnson — Fictional character in the Halo video game series.\n- Avery Jessup — Fictional character on the TV show 30 Rock (played by Elizabeth Banks).\n\nIf you tell me which Avery you have in mind (a person, a character from a specific show/book, or a brand

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

'It looks like you might mean Avery Lancaster, not Averi. Here’s a concise profile based on the info you provided:\n\n- Name: Avery Lancaster\n- Title: Co-Founder & Chief Executive Officer (CEO) of Insurellm\n- Location: San Francisco, California\n- Current Salary: $225,000\n- Insurellm Career Progression:\n  - 2015 – Present: Co-Founder & CEO\n  - 2013 – 2015: Senior Product Manager at Innovate Insurance Solutions\n- Additional notes: Avery is described as an innovative leader with risk management expertise, helping steer Insurellm to a leading position in the InsurTech space.\n\nThere’s a line in the notes about someone named Maxine being a Senior Data Engineer starting in 2021, which doesn’t align with Avery’s profile. It seems like a mix-up in the notes. If you meant a different person, or want me to verify against an official bio, I can help with that. Would you like me to correct the spelling to Avery and provide any additional details?'

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!